# Fine-tune a logo detector on Colab

Trains a YOLO model with `sports_analyzer.train` on a dataset built by `scripts/prepare_dataset.py`, then evaluates it on the dataset's validation split.

**Disconnects are fine.** Training output is saved to Google Drive after every epoch. If the session drops or you stop it, reconnect and run all cells again: training resumes after the last completed epoch, and a finished run is left alone.

Before running, zip the dataset locally and upload it to `MyDrive/sports-analyzer/datasets/` on Google Drive:

```bash
cd data/datasets && zip -r f1.zip f1 -x '*.cache'
```

Use a GPU runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/sports-analyzer"  # everything that must survive a disconnect lives here
BRANCH = "main"  # branch of the GitHub repo to install the code from
DATASET = "f1"  # DRIVE_DIR/datasets/f1.zip
MODEL = "yolo11s.pt"
EPOCHS = 150
IMGSZ = 1280
RUN_NAME = "f1_yolo11s_1280"  # an unfinished run with this name is resumed; change it to start a new one

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Same Ultralytics version as uv.lock, so the trained weights load locally
%pip install -q "sports-analyzer @ git+https://github.com/AndrewKM210/sports-analyser@{BRANCH}" ultralytics==8.4.174

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Copy the dataset from Drive to the Colab disk, which is much faster to read during training.

In [ ]:
import zipfile

zipfile.ZipFile(f"{DRIVE_DIR}/datasets/{DATASET}.zip").extractall("/content/datasets")

## Train

Starts the run, resumes it after a disconnect, or does nothing if it already finished.

In [ ]:
from sports_analyzer.train import train

run_dir = train(
    f"/content/datasets/{DATASET}/data.yaml",
    model=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,  # pick the largest batch that fits in GPU memory
    project=f"{DRIVE_DIR}/runs",
    name=RUN_NAME,
)

## Evaluate

Precision, recall and mAP per brand on the validation split, using the best epoch. Brands that `prepare_dataset.py` warned about (only in train, or only in val) can't be judged from this.

In [ ]:
from ultralytics import YOLO

YOLO(run_dir / "weights" / "best.pt").val(
    data=f"/content/datasets/{DATASET}/data.yaml",
    imgsz=IMGSZ,
    project=f"{DRIVE_DIR}/runs",
    name=f"{RUN_NAME}_val",
    exist_ok=True,
)

## Use the model locally

Download `best.pt` from `MyDrive/sports-analyzer/runs/<RUN_NAME>/weights/` and run:

```bash
uv run sports-analyzer analyze data/videos/f1.mp4 --sport f1 --detector finetuned --weights path/to/best.pt
```